In [2]:

!pip install langchain_community langchain langchain_text_splitters langchain_openai langchain_chroma


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.2/122.2 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 70.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.7/561.7 kB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 96.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 81.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2

In [14]:
import os
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from dotenv import load_dotenv


def load_documents(docs_path="/content/docs"):
  print("Loading Documents from Docs folder....")

  # check if directory esists
  if not os.path.exists(docs_path):
    raise FileNotFoundError(f"The folder{docs_path} does not exists")

  loader = DirectoryLoader(
        path=docs_path,
        glob="*.txt",
        loader_cls=TextLoader
    )

  documents = loader.load()

  if len(documents) == 0:
    raise FileNotFoundError("The file are empty")

  for i, doc in enumerate(documents):
    print(f"  Docuemnt {i+1}:")
    print(f"  Source: {doc.metadata['source']}")
    print(f"  Content: {len(doc.page_content)} characters")
    print(f"  Content preview: {doc.page_content[:100]}...")
    print(f"  metadata: {doc.metadata}")

  return documents


def split_docs(documents, chunk_size=1000, chunk_overlap=0):
  print("=========Chunking the docuemts ========")

  text_splliter = CharacterTextSplitter(
      chunk_size = chunk_size,
      chunk_overlap = chunk_overlap
  )

  chunks = text_splliter.split_documents(documents)

  if chunks:
    for i, chu in enumerate(chunks[:5]):
      print(f" Chunk: { i + 1}")
      print(f" Source: { chu.metadata['source']}")
      print(f" Context: {chu.page_content}")

  return chunks


def create_vector_store(chunks, persist_directory="/content/db/chroma_db"):
  print("========= Embedding Chunks =======")

  embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

  # create vector database
  vectorstore = Chroma.from_documents(
      documents = chunks,
      embedding= embedding_model,
      persist_directory=persist_directory,
      collection_metadata ={"hnsw:space": "cosine"}
  )

  print(f"Db created locally in {persist_directory}")
  return vectorstore


def main():
  print("============ Ingestion Pipeline =============")

  # 1. load the files
  documents = load_documents(docs_path="/content/docs")

  # 2. chunk the files
  chunks = split_docs(documents)

  # 3.embedding and store it in a vector DB
  vectorStore =  create_vector_store(chunks)

# if __name__ == "__main__":
main()

============ Ingestion Pipeline =============
Loading Documents from Docs folder....
  Docuemnt 1:
  Source: /content/docs/Microsoft.txt
  Content: 201014 characters
  Content preview: ﻿Microsoft
Microsoft Corporation is an American multinational Microsoft Corporation
corporation and ...
  metadata: {'source': '/content/docs/Microsoft.txt'}
  Docuemnt 2:
  Source: /content/docs/Google.txt
  Content: 232201 characters
  Content preview: ﻿Google
Google LLC (/ˈɡuːɡəl/ ⓘ , GOO-gəl) is an Google LLC
American multinational corporation and t...
  metadata: {'source': '/content/docs/Google.txt'}
=========Chunking the docuemts ========
 Chunk: 1
 Source: /content/docs/Microsoft.txt
 Context: ﻿Microsoft
Microsoft Corporation is an American multinational Microsoft Corporation
corporation and technology conglomerate
headquartered in Redmond, Washington.[2] Founded
in 1975, the company became influential in the rise of
personal computers through software like Windows,
and the company has since expa

OpenAIError: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.